# Tech Challenge Fase 2  
## Notebook 01.1 — Silver Alunos

### Responsabilidade do notebook

## Objetivo:
Importar as bibliotecas utilizadas durante o processo de auditoria e construção da camada Silver.

## Justificativa:
As bibliotecas são carregadas no início do notebook para ficarem disponíveis em  todas as etapas do pipeline.

## 1. Imports
## Ação:

Importa as bibliotecas necessárias para manipulação dos dados e acesso ao sistema de arquivos.

In [0]:
import json
import pandas as pd

from pathlib import Path

## 2. Definir a estrutura dos diretórios.

## Objetivo:
Definir as configurações utilizadas durante a execução do notebook.

## Justificativa:
Os caminhos são centralizados no config.json e na silver_metadata, garantindo consistência entre as camadas Bronze e Silver.

## Ação:
Carrega a configuração oficial do projeto e filtra os metadados da entidade processada neste notebook.

In [0]:
CONFIG_FILE_PATH = "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/config/config.json"

config = json.loads(
    Path(CONFIG_FILE_PATH).read_text(
        encoding="utf-8"
    )
)

BASE_PATH = Path(
    config["environment"]["base_path"]
)

CONFIG_PATH = Path(
    config["paths"]["config_path"]
)

LOG_PATH = Path(
    config["paths"]["log_path"]
)

EXECUTION_DATE = (
    config["project"]["execution_date"]
)

SILVER_METADATA_PATH = (
    CONFIG_PATH
    / "silver_metadata"
)

df_silver_metadata = pd.read_parquet(
    SILVER_METADATA_PATH
)

metadata_dataset = (
    df_silver_metadata[
        df_silver_metadata["dataset"]
        == "alunos"
    ]
    .sort_values("ano")
    .reset_index(drop=True)
)

if set(metadata_dataset["ano"]) != {2023, 2024, 2025}:
    raise ValueError(
        "Metadados Silver incompletos "
        "para o dataset alunos."
    )

CAMINHO_BRONZE = Path(
    metadata_dataset.iloc[0][
        "bronze_path"
    ]
).parent

CAMINHO_SILVER = Path(
    metadata_dataset.iloc[0][
        "silver_path"
    ]
).parent

print(
    "CAMINHO_BRONZE:",
    CAMINHO_BRONZE
)

print(
    "CAMINHO_SILVER:",
    CAMINHO_SILVER
)

display(metadata_dataset)

### Integração com o Silver Orquestrador

Este notebook não utiliza caminhos locais ou nomes de arquivos fixos.

Os caminhos da entidade `alunos` são obtidos da tabela:

```text
config/silver_metadata
```

A leitura é feita diretamente das partições Parquet da Bronze:

```text
bronze/alunos/ano=2023
bronze/alunos/ano=2024
bronze/alunos/ano=2025
```

A lógica de auditoria, limpeza e transformação construída originalmente permanece preservada.

## 3. Objetivo:
Carregar as partições anuais da entidade alunos na camada Bronze.

## Justificativa:
A Bronze foi persistida em Parquet, organizada por entidade e ano.
A Silver deve consumir diretamente essas partições governadas pelo silver_metadata.

## Ação:
Lê as partições Bronze referentes aos anos de 2023, 2024 e 2025.

In [0]:
def caminho_bronze_ano(ano):
    registro = metadata_dataset[
        metadata_dataset["ano"] == ano
    ].iloc[0]

    return Path(
        registro["bronze_path"]
    )


df_alunos_2023 = pd.read_parquet(
    caminho_bronze_ano(2023)
)

df_alunos_2024 = pd.read_parquet(
    caminho_bronze_ano(2024)
)

df_alunos_2025 = pd.read_parquet(
    caminho_bronze_ano(2025)
)

# 4. Auditoria da Fonte de Dados - Base Alunos

> **Nota**
>
> Durante o desenvolvimento deste projeto foi utilizado o dicionário oficial
> dos Microdados da Avaliação da Alfabetização disponibilizado pelo INEP como
> referência para interpretação das variáveis, domínios e regras de negócio
> presentes nas bases de dados.

## 4.1 Leitura das Bases

**Contexto**

Os microdados da Avaliação da Alfabetização foram organizados na camada
Bronze por entidade e particionados por ano, preservando a estrutura dos
arquivos disponibilizados pelo INEP.

Nesta etapa são carregadas as bases referentes aos anos de 2023, 2024 e
2025 para realização da auditoria estrutural e preparação da camada Silver.

**Objetivo**

Carregar as bases da entidade Alunos referentes aos anos de 2023, 2024 e
2025 para análise da estrutura e padronização dos dados.

**Resultado esperado**

Obter três DataFrames correspondentes às bases de alunos dos anos de 2023,
2024 e 2025, prontos para as etapas de auditoria e transformação.

In [0]:
# Objetivo:
# Carregar as partições anuais da
# entidade alunos na camada Bronze.

# Justificativa:
# A Bronze foi persistida em Parquet,
# organizada por entidade e ano.
# A Silver deve consumir diretamente
# essas partições governadas pelo
# silver_metadata.

# Ação:
# Lê as partições Bronze referentes
# aos anos de 2023, 2024 e 2025.

def caminho_bronze_ano(ano):
    registro = metadata_dataset[
        metadata_dataset["ano"] == ano
    ].iloc[0]

    return Path(
        registro["bronze_path"]
    )


df_alunos_2023 = pd.read_parquet(
    caminho_bronze_ano(2023)
)

df_alunos_2024 = pd.read_parquet(
    caminho_bronze_ano(2024)
)

df_alunos_2025 = pd.read_parquet(
    caminho_bronze_ano(2025)
)

## 4.2 Inspeção Inicial da Estrutura

**Contexto**

Após o carregamento das bases da camada Bronze, realiza-se uma inspeção
inicial para compreender a estrutura dos dados disponibilizados pelo INEP
em cada ano da avaliação.

Essa verificação permite identificar a quantidade de registros, colunas,
tipos de dados e possíveis diferenças estruturais entre as bases antes do
processo de padronização da camada Silver.

**Objetivo**

Inspecionar a estrutura das bases de alunos dos anos de 2023, 2024 e 2025,
identificando eventuais diferenças que possam impactar as etapas seguintes
do pipeline.

**Resultado esperado**

Obter uma visão inicial da estrutura das bases, permitindo identificar
alterações entre os anos e subsidiar as etapas de auditoria e padronização.

In [0]:
# Objetivo:
# Realizar uma inspeção inicial da
# estrutura das bases de alunos.

# Justificativa:
# A inspeção inicial permite verificar
# o esquema das bases e identificar
# possíveis alterações nas colunas
# disponibilizadas pelo INEP ao longo
# dos anos da avaliação.

# Ação:
# Exibe a estrutura das bases de
# alunos para comparação entre os
# anos de 2023, 2024 e 2025.

print(df_alunos_2023.columns.tolist())

print(df_alunos_2024.columns.tolist())

print(df_alunos_2025.columns.tolist())

## 4.3 Comparação dos Tipos de Dados

**Contexto**

Além da estrutura das colunas, é importante verificar se os tipos de dados
foram mantidos entre as bases de 2023, 2024 e 2025.

Diferenças de tipagem podem comprometer a padronização da camada Silver e
dificultar a integração das partições durante a construção da camada Gold.

**Objetivo**

Comparar os tipos de dados das bases de alunos dos anos de 2023, 2024 e
2025, identificando possíveis divergências estruturais.

**Resultado esperado**

Obter um relatório de compatibilidade dos tipos de dados, permitindo
identificar eventuais diferenças que necessitem de padronização antes da
persistência da camada Silver.

In [0]:
# Objetivo:
# Comparar os tipos de dados das
# bases de alunos dos anos de
# 2023, 2024 e 2025.

# Justificativa:
# Colunas com o mesmo nome podem
# apresentar tipos diferentes entre
# os anos ou novas variáveis podem
# ter sido incorporadas pelo INEP.

# Ação:
# Consolida os tipos de dados das
# três bases em um único relatório
# e classifica a compatibilidade
# entre os esquemas.

relatorio_dtypes = pd.DataFrame({
    "2023": df_alunos_2023.dtypes.astype(str),
    "2024": df_alunos_2024.dtypes.astype(str),
    "2025": df_alunos_2025.dtypes.astype(str)
})

relatorio_dtypes = relatorio_dtypes.replace("nan", pd.NA)


def classificar_status(linha):

    if (
        pd.notna(linha["2023"])
        and pd.isna(linha["2024"])
        and pd.isna(linha["2025"])
    ):
        return "Exclusiva de 2023"

    if (
        pd.isna(linha["2023"])
        and pd.notna(linha["2024"])
        and pd.notna(linha["2025"])
    ):
        return "Nova a partir de 2024"

    if (
        pd.isna(linha["2023"])
        and pd.isna(linha["2024"])
        and pd.notna(linha["2025"])
    ):
        return "Nova em 2025"

    tipos = linha.dropna()

    if len(tipos.unique()) == 1:
        return "Compatível"

    return "Divergente"


relatorio_dtypes["status"] = relatorio_dtypes.apply(
    classificar_status,
    axis=1
)

display(relatorio_dtypes)

## 4.4 Análise dos Valores Ausentes

**Contexto**

Após a verificação da estrutura e dos tipos de dados, torna-se necessário
avaliar a presença de valores ausentes nas bases dos diferentes anos.

Essa análise permite identificar possíveis impactos sobre a qualidade dos
dados e definir se será necessário realizar algum tratamento antes da
persistência da camada Silver.

**Objetivo**

Identificar e quantificar a ocorrência de valores ausentes nas bases da
entidade Alunos, avaliando a necessidade de tratamento durante o processo
de transformação.

**Resultado esperado**

Obter um diagnóstico da completude dos dados, permitindo justificar as
decisões adotadas em relação ao tratamento de valores ausentes na camada
Silver.

In [0]:
# Objetivo:
# Analisar a ocorrência de valores
# ausentes nas bases da entidade
# Alunos.

# Justificativa:
# A identificação de valores ausentes
# permite avaliar a qualidade dos
# dados e definir eventuais ações de
# tratamento antes da persistência da
# camada Silver.

# Ação:
# Calcula a quantidade de valores
# ausentes por coluna para cada ano
# da avaliação.

for ano, df in [
    (2023, df_alunos_2023),
    (2024, df_alunos_2024),
    (2025, df_alunos_2025)
]:

    print(f"\nValores ausentes - {ano}")

    valores_ausentes = (
        df.isna()
          .sum()
          .loc[lambda s: s > 0]
          .sort_values(ascending=False)
          .to_frame("Valores Ausentes")
    )

    if valores_ausentes.empty:
        print("Nenhum valor ausente encontrado.")
    else:
        display(valores_ausentes)

### 4.4.1 Investigação das Ocorrências

**Contexto**

A análise inicial identificou a ocorrência de valores ausentes em algumas
colunas da Base Alunos.

Antes de definir qualquer estratégia de tratamento, torna-se necessário
investigar a origem desses registros, verificando se os valores ausentes
representam inconsistências na base ou se decorrem de regras de negócio
adotadas pelo INEP.

**Objetivo**

Investigar a origem dos valores ausentes identificados na Base Alunos,
subsidiando as decisões de tratamento adotadas na camada Silver.

**Resultado esperado**

Identificar o padrão dos registros com valores ausentes e documentar a
estratégia de tratamento mais adequada para cada situação encontrada.

In [0]:
# Objetivo:
# Investigar os registros de 2025 que
# apresentam ausência de informações
# de identificação da escola e do
# município.

# Justificativa:
# A ocorrência do mesmo número de
# valores ausentes em diferentes
# colunas pode indicar um padrão de
# negócio ou uma inconsistência na
# base de origem.

# Ação:
# Exibe uma amostra dos registros que
# possuem município não informado para
# análise das demais variáveis.

df_alunos_2025[
    df_alunos_2025["CO_MUNICIPIO"].isna()
].head(20)

In [0]:
# Objetivo:
# Investigar a origem dos valores
# ausentes nas colunas de peso e
# proficiência da avaliação.

# Justificativa:
# A ocorrência de valores ausentes
# pode decorrer de regras de negócio
# do INEP e não necessariamente de
# inconsistências na base de dados.

# Ação:
# Analisa os registros que possuem
# proficiência não informada,
# verificando o comportamento das
# demais variáveis relacionadas à
# realização da avaliação.

df_alunos_2025[
    df_alunos_2025["VL_PROFICIENCIA_LP"].isna()
][[
    "IN_PRESENCA_LP",
    "IN_PREENCHIMENTO_LP",
    "VL_PESO_ALUNO_LP",
    "VL_PROFICIENCIA_LP",
    "IN_ALFABETIZADO"
]].value_counts(dropna=False)

### 4.4.2 Análise dos Resultados

A investigação demonstrou que os valores ausentes identificados nas colunas
**VL_PESO_ALUNO_LP** e **VL_PROFICIENCIA_LP** não representam inconsistências
na base de dados.

Esses registros estão associados a estudantes que não participaram da
avaliação (`IN_PRESENCA_LP = 0`) ou que, embora presentes, não realizaram o
preenchimento da prova (`IN_PREENCHIMENTO_LP = 0`). Nessas situações, a
ausência de peso amostral e de proficiência constitui uma característica
esperada dos microdados disponibilizados pelo INEP.

Também foram identificados registros da base de 2025 com ausência das
informações de escola e município. Como esses estudantes possuem identificação,
participação na avaliação e, em sua maioria, resultados válidos, optou-se por
preservar esses registros sem imputação dos valores ausentes, mantendo a
fidelidade aos dados de origem.

Dessa forma, não foram aplicadas técnicas de preenchimento ou remoção de
valores ausentes, uma vez que os casos identificados decorrem de regras de
negócio da base de origem e não de inconsistências no processo de tratamento
da camada Silver.

## 4.5 Seleção das Colunas da Camada Silver

**Contexto**

A auditoria da estrutura das bases mostrou que o INEP ampliou a quantidade
de informações disponibilizadas em 2025, incorporando colunas relacionadas
aos blocos da avaliação e aos respectivos gabaritos e respostas dos
estudantes.

Como essas informações não fazem parte do escopo analítico deste projeto,
elas não serão mantidas na camada Silver.

Nesta etapa são selecionadas apenas as colunas necessárias para a construção
da Base Alunos da camada Silver, padronizando a estrutura das bases dos anos
de 2023, 2024 e 2025.

**Objetivo**

Definir o conjunto de atributos que comporá a Base Alunos da camada Silver,
preservando apenas as informações relevantes para as análises propostas no
Tech Challenge.

**Resultado esperado**

Obter três bases com a mesma estrutura de colunas, aptas para a persistência
particionada da camada Silver e para posterior integração na camada Gold.

In [0]:
# Objetivo:
# Selecionar as colunas que farão
# parte da Base Alunos da camada
# Silver.

# Justificativa:
# As bases de 2025 incorporaram
# novas colunas relacionadas aos
# blocos da avaliação, que não
# fazem parte do escopo analítico
# deste projeto.

# Ação:
# Seleciona o conjunto de colunas
# que comporá a Base Alunos da
# camada Silver, padronizando a
# estrutura entre os anos.

colunas_silver = [
    "NU_ANO_AVALIACAO",
    "CO_UF",
    "SG_UF",
    "ID_ALUNO",
    "TP_SERIE",
    "ID_ESCOLA",
    "TP_DEPENDENCIA",
    "CO_MUNICIPIO",
    "NO_MUNICIPIO",
    "IN_PRESENCA_LP",
    "IN_PREENCHIMENTO_LP",
    "CO_CADERNO_LP",
    "VL_PESO_ALUNO_LP",
    "VL_PROFICIENCIA_LP",
    "IN_ALFABETIZADO"
]

df_alunos_2023 = df_alunos_2023[colunas_silver].copy()

df_alunos_2024 = df_alunos_2024[colunas_silver].copy()

df_alunos_2025 = df_alunos_2025[colunas_silver].copy()

## 4.6 Validação da Estrutura da Camada Silver

**Contexto**

Após a seleção das colunas que comporão a Base Alunos da camada Silver,
torna-se necessário validar se as três bases apresentam a mesma estrutura.

Essa verificação garante que todas as partições anuais compartilhem o mesmo
esquema, permitindo sua integração na camada Gold sem necessidade de
transformações adicionais.

**Objetivo**

Validar a estrutura das bases de alunos após a seleção das colunas da camada
Silver.

**Resultado esperado**

Obter três bases com a mesma quantidade de colunas e estrutura compatível,
prontas para a persistência particionada da camada Silver.

In [0]:
# Objetivo:
# Validar a estrutura das bases
# após a seleção das colunas da
# camada Silver.

# Justificativa:
# Todas as partições anuais devem
# possuir o mesmo esquema antes da
# persistência na camada Silver.

# Ação:
# Compara a quantidade de colunas
# das bases de 2023, 2024 e 2025.

validacao_colunas = pd.DataFrame({
    "Base": ["2023", "2024", "2025"],
    "Quantidade de Colunas": [
        len(df_alunos_2023.columns),
        len(df_alunos_2024.columns),
        len(df_alunos_2025.columns)
    ]
})

validacao_colunas["Status"] = (
    "Compatível"
    if validacao_colunas["Quantidade de Colunas"].nunique() == 1
    else "Divergente"
)

display(validacao_colunas)

## 4.7 Persistência da Base Alunos

**Contexto**

Após a auditoria estrutural, seleção das colunas e validação do esquema,
as bases da entidade Alunos estão preparadas para serem persistidas na
camada Silver.

Seguindo a arquitetura definida para este projeto, os dados são armazenados
de forma particionada por ano, preservando a organização do Data Lake e
facilitando futuras integrações na camada Gold.

**Objetivo**

Persistir as bases de alunos da camada Silver em partições anuais,
preservando a padronização estrutural obtida durante o processo de
transformação.

**Resultado esperado**

Obter três partições da Base Alunos na camada Silver, correspondentes aos
anos de 2023, 2024 e 2025, prontas para consumo analítico e integração na
camada Gold.

In [0]:
# Objetivo:
# Persistir as bases da entidade
# alunos na camada Silver.

# Justificativa:
# A persistência utiliza os caminhos
# e nomes de arquivo registrados na
# silver_metadata, eliminando caminhos
# fixos e mantendo o particionamento
# anual definido no setup.

# Ação:
# Grava as bases tratadas dos anos de
# 2023, 2024 e 2025 em CSV UTF-8.

for ano, df in [
    (2023, df_alunos_2023),
    (2024, df_alunos_2024),
    (2025, df_alunos_2025)
]:
    registro = metadata_dataset[
        metadata_dataset["ano"] == ano
    ].iloc[0]

    destino = Path(
        registro["silver_path"]
    )

    nome_arquivo = registro[
        "silver_file_name"
    ]

    destino.mkdir(
        parents=True,
        exist_ok=True
    )

    df.to_csv(
        destino / nome_arquivo,
        sep=";",
        decimal=",",
        encoding="utf-8",
        index=False
    )

    print(
        f"Silver salva: "
        f"{destino / nome_arquivo}"
    )

# Conclusão

Ao longo deste notebook foi realizada a auditoria estrutural das bases da
entidade **Alunos** referentes aos anos de **2023**, **2024** e
**2025**, identificando a evolução do esquema de dados disponibilizado pelo
INEP.

A auditoria contemplou a verificação da estrutura das bases, a comparação
dos tipos de dados e a análise dos valores ausentes. A investigação
realizada demonstrou que os valores ausentes identificados decorrem das
regras de negócio dos microdados disponibilizados pelo INEP, não sendo
necessário aplicar técnicas de imputação ou exclusão de registros durante o
processo de construção da camada Silver.

A análise também evidenciou que, a partir de 2025, foram incorporadas novas
colunas relacionadas aos blocos da avaliação de Língua Portuguesa.
Considerando os objetivos analíticos definidos para este projeto, optou-se
por não preservar essas informações na camada Silver, mantendo apenas os
atributos necessários para as análises e etapas posteriores do pipeline.

Após a padronização das colunas e validação da estrutura, as bases foram
persistidas na camada **Silver**, mantendo a organização por entidade e o
particionamento por ano, conforme a arquitetura definida para o projeto.

Dessa forma, a Base Alunos da camada Silver passa a representar uma versão
padronizada, auditada e organizada dos microdados da avaliação,
constituindo uma fonte confiável para a construção dos indicadores,
agregações e análises desenvolvidas nas etapas posteriores da camada Gold.